<a href="https://colab.research.google.com/github/ikeyareinaldosae/ITC501---Artificial-Intelligence-Applications/blob/main/Milestone_1_Brief_AI_for_Tabular_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# 0. GATHERING IMPORTS
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import lightgbm as lgb  # Efficient gradient boosting for 1M rows

In [8]:
# ==========================================
# 1. DATA SOURCING
# ==========================================

# Assuming the downloaded Kaggle dataset is saved locally as 'customer_churn_1M.csv'
df = pd.read_csv('customer_churn_1M.csv')

# If not locally saved we can mount the drive and read it from there.
#from google.colab import drive
#drive.mount('/content/drive')
#data_folder = "drive/MyDrive/datasets"
#data = pd.read_csv(f'{data_folder}/customer_churn_1M.csv')
#data

# Inspecting the target distribution to check for class imbalance
print(f"Dataset Shape: {df.shape}")
print(f"Churn Distribution:\n{df['churn'].value_counts(normalize=True)}")

Dataset Shape: (1000000, 32)
Churn Distribution:
churn
0    0.900773
1    0.099227
Name: proportion, dtype: float64


In [9]:
# ==========================================
# 2. DATA CLEANING & PREPROCESSING
# ==========================================

print("\nCleaning data...")

# Drop structural identifiers that won't help prediction
id_cols = ['customer_id', 'name']  # Adjust based on exact schema if names differ
df = df.drop(columns=[col for col in id_cols if col in df.columns])

# Handle Missing Values
# Numerical: Fill with median to protect against outliers
num_cols = df.select_dtypes(include=[np.number]).columns.drop('churn', errors='ignore')
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical: Fill with 'Unknown'
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')

# Outlier Mitigation: Cap extreme behavioral anomalies at the 99th percentile
# (e.g., highly inflated service calls or complaints due to system bugs)
cap_cols = ['num_service_calls', 'num_complaints', 'late_payments']
for col in cap_cols:
    if col in df.columns:
        upper_limit = df[col].quantile(0.99)
        df[col] = np.where(df[col] > upper_limit, upper_limit, df[col])


Cleaning data...


In [10]:
# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
print("Engineering features...")

# Dynamic interaction features to capture customer friction points
if 'num_service_calls' in df.columns and 'num_complaints' in df.columns:
    # Total friction touchpoints
    df['total_friction_events'] = df['num_service_calls'] + df['num_complaints']

if 'annual_income' in df.columns and 'monthly_charges' in df.columns:
    # Financial commitment ratio: percentage of income going to this service
    df['income_absorption_rate'] = (df['monthly_charges'] * 12) / (df['annual_income'] + 1)

# Separate features (X) and target (y)
X = df.drop(columns=['churn'])
y = df['churn']

# Update numerical and categorical column lists after engineering
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Define Preprocessing Transformers (Standard Scaling for numeric, One-Hot for text)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_features)
    ])

# Split into train and test sets (80/20) with stratification to keep churn ratios even
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Apply preprocessing transformations
print("Transforming arrays...")
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

Engineering features...
Transforming arrays...


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [11]:
# ==========================================
# 4. MODEL SELECTION & TRAINING
# ==========================================
print("\nTraining models...")

# Model A: Baseline Logistic Regression (Fast, linear benchmark)
print("Training Baseline Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_proc, y_train)
lr_preds = lr_model.predict(X_test_proc)

# Model B: LightGBM (Handles non-linear relationships, scale-invariant, highly accurate)
print("Training LightGBM Classifier...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train_proc, y_train)
lgb_preds = lgb_model.predict(X_test_proc)


Training models...
Training Baseline Logistic Regression...
Training LightGBM Classifier...
[LightGBM] [Info] Number of positive: 79382, number of negative: 720618
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.076899 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1784
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.099227 -> initscore=-2.205838
[LightGBM] [Info] Start training from score -2.205838


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
# ==========================================
# 5. EVALUATION
# ==========================================
def print_evaluation_metrics(model_name, y_true, y_pred):
    print(f"\n================ {model_name} EVALUATION ================")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}  (When it predicts churn, how often is it right?)")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}     (Of all real churners, how many did we flag?)")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}     (Harmonic balance of precision & recall)")
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred))

# Print evaluations
print_evaluation_metrics("Logistic Regression", y_test, lr_preds)
print_evaluation_metrics("LightGBM Classifier", y_test, lgb_preds)


================ Logistic Regression EVALUATION ================
Accuracy:  0.9009
Precision: 0.7059  (When it predicts churn, how often is it right?)
Recall:    0.0018     (Of all real churners, how many did we flag?)
F1-Score:  0.0036     (Harmonic balance of precision & recall)

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95    180155
           1       0.71      0.00      0.00     19845

    accuracy                           0.90    200000
   macro avg       0.80      0.50      0.48    200000
weighted avg       0.88      0.90      0.85    200000


================ LightGBM Classifier EVALUATION ================
Accuracy:  0.9008
Precision: 0.6383  (When it predicts churn, how often is it right?)
Recall:    0.0015     (Of all real churners, how many did we flag?)
F1-Score:  0.0030     (Harmonic balance of precision & recall)

Detailed Classification Report:
              precision    recall  f1-sco